In [2]:
import os
import cv2
import pickle
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

# --- STEP 1: PERFORMANCE CONFIG (CPU Focus for now) ---
# We disable GPU visibility for this specific indexing step to save resources
os.environ["CUDA_VISIBLE_DEVICES"] = "-1" 
print("Running in CPU-only mode for indexing...")

# --- STEP 2: GLOBAL VARIABLES & PATHS ---
# Path to the dataset in Kaggle
DATASET_ROOT = "/kaggle/input/vimeo-90k-1/vimeo_settuplet_1/sequences"
# Path to the triplet model we'll eventually transfer weights from
PRETRAINED_MODEL_PATH = "/kaggle/input/vfi-epoch-99-keras/keras/default/1/vfi_epoch_99.keras"
# Our persistence file
PICKLE_PATH = "septuplets.pkl"
# Where to save training checkpoints
OUTPUT_DIR = "/kaggle/working/checkpoints"

# Vimeo default res is 448x256
IMAGE_SIZE = (448, 256) 
CROP_SIZE = (256, 256)   

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

Running in CPU-only mode for indexing...


## Creating Persistence with Pickle

In [3]:
# --- STEP 3: DATA INDEXING (PHASE 1) ---
def generate_septuplet_pickle(dataset_path, output_pkl):
    """
    Walks through the vimeo-90k-1 directory structure once to index all septuplets.
    Structure: sequences / {video_id} / {folder_id} / im1.png...im7.png
    """
    if os.path.exists(output_pkl):
        print(f"✅ Index found at {output_pkl}. Loading...")
        with open(output_pkl, 'rb') as f:
            data = pickle.load(f)
            print(f"Loaded {len(data)} septuplet sequences.")
            return data
    
    print("🔍 No index found. Walking directory (this may take a few minutes)...")
    all_septuplets = []
    
    if not os.path.exists(dataset_path):
        print(f"❌ Error: Dataset path {dataset_path} not found!")
        return []

    # Iterate through video folders (e.g., 00001, 00002...)
    video_folders = sorted(os.listdir(dataset_path))
    for video_id in video_folders:
        video_path = os.path.join(dataset_path, video_id)
        if not os.path.isdir(video_path): continue
        
        # Iterate through sequence folders (e.g., 0001, 0002...)
        seq_folders = sorted(os.listdir(video_path))
        for seq_id in seq_folders:
            seq_path = os.path.join(video_path, seq_id)
            
            # Construct the list of 7 image paths for this septuplet
            septuplet_paths = [os.path.join(seq_path, f"im{i}.png") for i in range(1, 8)]
            
            # Verify integrity: ensure the sequence folder is complete
            # We check the first, middle, and last to be fast
            if os.path.exists(septuplet_paths[0]) and \
               os.path.exists(septuplet_paths[3]) and \
               os.path.exists(septuplet_paths[6]):
                all_septuplets.append(septuplet_paths)
                
    print(f"✨ Found {len(all_septuplets)} septuplet sequences.")
    
    # Save the list to the pickle file for future sessions
    with open(output_pkl, 'wb') as f:
        pickle.dump(all_septuplets, f)
    print(f"💾 septuplets.pkl saved to current directory.")
    
    return all_septuplets

# Execute indexing logic
SEPTUPLET_LIST = generate_septuplet_pickle(DATASET_ROOT, PICKLE_PATH)

print("\n--- SETUP COMPLETE ---")
print(f"Total sequences indexed: {len(SEPTUPLET_LIST)}")
if len(SEPTUPLET_LIST) > 0:
    print(f"Sample path: {SEPTUPLET_LIST[0][0]}")

🔍 No index found. Walking directory (this may take a few minutes)...
✨ Found 9953 septuplet sequences.
💾 septuplets.pkl saved to current directory.

--- SETUP COMPLETE ---
Total sequences indexed: 9953
Sample path: /kaggle/input/vimeo-90k-1/vimeo_settuplet_1/sequences/00001/0001/im1.png
